# Analyse: Kameradistanz


In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from transformers import AutoFeatureExtractor, AutoImageProcessor, AutoModelForImageClassification


In [15]:
DATA_DIR = Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '05_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_camera_distance.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'

SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60
MODEL_NAME = 'pszemraj/dinov2-small-film-shot-classifier'
UNKNOWN_LABEL = 'Unbestimmt'
MIN_CONFIDENCE = 0.50

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Running on device: {device}')
df = pd.read_csv(COMBINED_CSV)


Running on device: cpu


In [16]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'
df[video_id_col] = df[video_id_col].astype(str)

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [17]:
try:
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
except Exception:
    processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(MODEL_NAME).to(device).eval()
id2label = {int(k): v for k, v in model.config.id2label.items()}
print('Loaded model:', MODEL_NAME)
print('Model classes:', id2label)

def get_duration_seconds(row: pd.Series):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, default_max_frames, len(frame_files)))
    indices = np.linspace(0, len(frame_files) - 1, num=target_frames, dtype=int)
    return [frame_files[idx] for idx in np.unique(indices)]

def predict_frame_shot_scale(image_path: Path):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits[0]
            probs = F.softmax(logits, dim=-1)

        idx = int(torch.argmax(probs).item())
        conf = float(probs[idx].item())
        label = str(id2label.get(idx, idx))

        if conf < MIN_CONFIDENCE:
            return UNKNOWN_LABEL, np.nan
        return label, conf
    except Exception as exc:
        print(f'Error on {image_path}: {exc}')
        return UNKNOWN_LABEL, np.nan


Loaded model: pszemraj/dinov2-small-film-shot-classifier
Model classes: {0: 'ambiguous', 1: 'closeUp', 2: 'detail', 3: 'extremeLongShot', 4: 'fullShot', 5: 'longShot', 6: 'mediumCloseUp', 7: 'mediumShot'}


In [18]:
result_rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    video_id = str(row.get(video_id_col))
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(video_id, duration_seconds=duration_seconds)

    labels = []
    confs = []

    for frame_path in frame_paths:
        label, conf = predict_frame_shot_scale(frame_path)
        if label != UNKNOWN_LABEL:
            labels.append(label)
            if not np.isnan(conf):
                confs.append(conf)

    metrics = {
        'video_id': video_id,
        'camera_distance_label': UNKNOWN_LABEL,
        'camera_distance_confidence': np.nan,
        'camera_distance_unique_labels': np.nan,
        'processed_frame_count': len(frame_paths),
    }

    if labels:
        counts = Counter(labels)
        top_label, _ = counts.most_common(1)[0]
        metrics['camera_distance_label'] = top_label
        metrics['camera_distance_unique_labels'] = len(counts)

        top_confs = [
            c for l, c in zip(labels, confs + [np.nan] * max(0, len(labels) - len(confs)))
            if l == top_label and not np.isnan(c)
        ]
        metrics['camera_distance_confidence'] = float(np.mean(top_confs)) if top_confs else (float(np.mean(confs)) if confs else np.nan)

    result_rows.append(metrics)

feature_df = pd.DataFrame(result_rows)
merged = df.merge(feature_df, on='video_id', how='left') if 'video_id' in df.columns else df.merge(feature_df, left_on=video_id_col, right_on='video_id', how='left')

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_CSV} with shape {merged.shape}')


Processing videos: 100%|██████████| 500/500 [14:15<00:00,  1.71s/it]

Wrote ..\..\data\04_analysis_results\visual_features\05_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_camera_distance.csv with shape (500, 49)


In [19]:
if 'influencer_type' in merged.columns:
    print(merged[['influencer_type', 'camera_distance_confidence']].groupby('influencer_type').describe())
    print(merged.groupby(['influencer_type', 'camera_distance_label']).size().unstack(fill_value=0))

preview_cols = [video_id_col, 'camera_distance_label', 'camera_distance_confidence', 'camera_distance_unique_labels', 'processed_frame_count']
preview_cols = [c for c in preview_cols if c in merged.columns]
merged[preview_cols].head()


                camera_distance_confidence                                \
                                     count      mean       std       min   
influencer_type                                                            
ai                                   250.0  0.770239  0.105253  0.542862   
real                                 250.0  0.758118  0.105587  0.512412   

                                                         
                      25%       50%       75%       max  
influencer_type                                          
ai               0.685153  0.776798  0.846516  0.986351  
real             0.677817  0.770816  0.830410  0.989110  
camera_distance_label  ambiguous  closeUp  detail  extremeLongShot  fullShot  \
influencer_type                                                                
ai                             1       19       2                1        23   
real                           2       23       1               11        20   

camera_d

,video_id,camera_distance_label,camera_distance_confidence,camera_distance_unique_labels,processed_frame_count
0,7516243181650988334,mediumShot,0.833568,1,7
1,7515925552549678378,mediumCloseUp,0.688760,1,10
2,7521159314757684535,mediumShot,0.797076,2,8
3,7521486299098778894,mediumShot,0.974943,3,8
4,7521490969468865847,closeUp,0.784160,2,8
